# BugSight QA - training on Colab (GPU)

Trains YOLOv8s on the semi-synthetic dataset (`data/processed_semi`) with
`configs/train_semi_gpu.yaml`. About 20-30 min on a free T4.

**Before you start**
1. On your PC: `python scripts/make_colab_bundle.py` -> `dist/bugsight_colab.zip` (~125 MB).
2. Upload `bugsight_colab.zip` to the root of your Google Drive (`MyDrive/`), or to this
   session with the Files panel (use `--split-mb 9` and upload all `.partNN` files if your
   uploader has a size limit).
3. Here: *Runtime > Change runtime type > T4 GPU*, then *Runtime > Run all*.

At the end the run folder (weights, curves, per-class report) is copied to
`MyDrive/bugsight_runs/` and downloaded as a zip.

In [ ]:
# 1. Check the GPU (must list a Tesla T4 or similar)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get the bundle. Looked up in this order:
#    a) split parts /content/bugsight_colab.zip.part00, part01, ... (Files panel upload)
#    b) /content/bugsight_colab.zip
#    c) MyDrive/bugsight_colab.zip on Google Drive
import glob
import shutil
from pathlib import Path

ZIP = '/content/bugsight_colab.zip'
USE_DRIVE = False
parts = sorted(glob.glob(ZIP + '.part*'))
if parts:
    with open(ZIP, 'wb') as out:
        for part in parts:
            with open(part, 'rb') as f:
                shutil.copyfileobj(f, out)
    print(f'Joined {len(parts)} parts')
elif not Path(ZIP).exists():
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP = '/content/drive/MyDrive/bugsight_colab.zip'
    USE_DRIVE = True
print('Bundle:', ZIP, f'{Path(ZIP).stat().st_size / 1e6:.0f} MB')


In [ ]:
# 3. Unzip and install Ultralytics
!rm -rf /content/bugsight-qa && mkdir -p /content/bugsight-qa
!unzip -q "{ZIP}" -d /content/bugsight-qa
%cd /content/bugsight-qa
!pip install -q ultralytics

In [ ]:
# 4. Point data.yaml at the Colab copy of the dataset (it holds a Windows path)
import re
from pathlib import Path

yaml_path = Path('data/processed_semi/data.yaml')
text = yaml_path.read_text()
text = re.sub(r'^path:.*$', f'path: "{yaml_path.parent.resolve().as_posix()}"', text, flags=re.M)
yaml_path.write_text(text)
print(text)
for split in ('train', 'val', 'test'):
    print(split, len(list(Path(f'data/processed_semi/images/{split}').glob('*.jpg'))), 'images')

In [ ]:
# 5. Train (then the per-class evaluation on the test split runs automatically)
!python scripts/train.py --config configs/train_semi_gpu.yaml

In [ ]:
# 6. Results: per-class report and training curves
from pathlib import Path
from IPython.display import Image, Markdown, display

best = max(Path('runs').rglob('bugsight_yolov8s_semi*/weights/best.pt'), key=lambda p: p.stat().st_mtime)
run_dir = best.parent.parent
print('Best weights:', best)
for report in sorted(Path('runs/reports').glob(f'{run_dir.name}_*.md')):
    display(Markdown(report.read_text()))
for name in ('results.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg'):
    if (run_dir / name).exists():
        display(Image(filename=str(run_dir / name), width=900))

In [ ]:
# 7. Save the run to Drive and download it
import shutil

for report in Path('runs/reports').glob(f'{run_dir.name}_*'):
    shutil.copy(report, run_dir)
out = shutil.make_archive(f'/content/{run_dir.name}', 'zip', root_dir=run_dir.parent, base_dir=run_dir.name)
if USE_DRIVE:
    dest = Path('/content/drive/MyDrive/bugsight_runs')
    dest.mkdir(exist_ok=True)
    shutil.copy(out, dest)
    print('Copied to', dest / Path(out).name)
from google.colab import files
files.download(out)

## Use the model on your PC

Unzip the downloaded `bugsight_yolov8s_semi.zip` into
`bugsight-qa/runs/detect/runs/` so that
`runs/detect/runs/bugsight_yolov8s_semi/weights/best.pt` exists. The API
(`python scripts/serve.py`) picks it up automatically and `/api/metrics` reads its `results.csv`.